In [13]:
%%writefile parallel_sum.cpp
#include <iostream>
#include <vector>
#include <omp.h>
#include <iomanip>

// Normal Grain Size: Large enough to outweigh the "Burden"
#define GRAIN_SIZE 100000

long long parallel_sum(int* arr, int n) {
    // Base Case: If the chunk is small, sum it serially
    if (n <= GRAIN_SIZE) {
        long long sum = 0;
        for (int i = 0; i < n; i++) sum += arr[i];
        return sum;
    }

    // Divide: Split the array in half
    int mid = n / 2;
    long long x = 0, y = 0;

    // Fork: Spawn a task for the first half
    #pragma omp task shared(x)
    x = parallel_sum(arr, mid);

    // Continue: Current thread handles the second half
    y = parallel_sum(arr + mid, n - mid);

    // Join: Wait for the spawned task to finish
    #pragma omp taskwait

    return x + y;
}

double run_experiment(int n, int threads) {
    std::vector<int> arr(n, 1); // Fill array with 1s for easy checking
    long long total = 0;
    omp_set_num_threads(threads);

    double start = omp_get_wtime();
    #pragma omp parallel
    {
        #pragma omp single
        {
            total = parallel_sum(arr.data(), n);
        }
    }
    return omp_get_wtime() - start;
}

int main() {
    int n = 1 << 26; // ~67 Million elements
    int thread_counts[] = {1, 2, 4, 8, 16, 32, 64, 128};

    std::cout << std::fixed << std::setprecision(4);
    std::cout << "Threads\tTime(s)\t\tSpeedup\t\tEfficiency\n";

    // Baseline: Measure T1
    double T1 = run_experiment(n, 1);

    for (int i = 0; i < 8; i++) {
        int P = thread_counts[i];
        double TP = run_experiment(n, P);

        // Ensure 100% baseline for the first row
        if (P == 1) TP = T1;

        double speedup = T1 / TP;
        double efficiency = (speedup / P) * 100;

        std::cout << P << "\t" << TP << "\t" << speedup << "\t\t" << efficiency << "%\n";
    }
    return 0;
}

Overwriting parallel_sum.cpp


In [14]:
!g++ -fopenmp -O3 parallel_sum.cpp -o parallel_sum
!./parallel_sum

Threads	Time(s)		Speedup		Efficiency
1	0.0344	1.0000		100.0000%
2	0.0282	1.2210		61.0483%
4	0.0282	1.2193		30.4824%
8	0.0284	1.2124		15.1553%
16	0.0292	1.1800		7.3748%
32	0.0297	1.1598		3.6244%
64	0.0457	0.7542		1.1784%
128	0.0487	0.7068		0.5522%
